In [6]:
from scipy.io import loadmat
import numpy as np
import pandas as pd

data = loadmat('./influenza_outbreak_dataset.mat')

# print(data.keys())
for key, val in data.items():
    if not key.startswith('__'):  # filter out metadata keys, these are generated by matlab and dont contain useful info for us
        print(f"Variable: {key} | Type: {type(val)} | Shape: {getattr(val, 'shape', 'N/A')}")


# Finding shape of training and testing data given already in the dataset

X_tr = data['flu_X_tr'][0, 0]     
Y_tr = data['flu_Y_tr'][0, 0]

X_te = data['flu_X_te'][0, 0]
Y_te = data['flu_Y_te'][0, 0]

print("\nTrain Features (X_tr) shape:", X_tr.shape)  # e.g., (N_test_days, 545)
print("Train Target (Y_tr) shape:", Y_tr.shape)      # e.g., (N_test_days, 1)

print("\nTest Features (X_te) shape:", X_te.shape)  
print("Test Target (Y_te) shape:", Y_te.shape)   


#  

Variable: flu_X_tr | Type: <class 'numpy.ndarray'> | Shape: (1, 48)
Variable: flu_Y_tr | Type: <class 'numpy.ndarray'> | Shape: (1, 48)
Variable: flu_X_te | Type: <class 'numpy.ndarray'> | Shape: (1, 48)
Variable: flu_Y_te | Type: <class 'numpy.ndarray'> | Shape: (1, 48)
Variable: flu_locs | Type: <class 'numpy.ndarray'> | Shape: (1, 48)
Variable: flu_keywords | Type: <class 'numpy.ndarray'> | Shape: (1, 525)

Train Features (X_tr) shape: (1095, 545)
Train Target (Y_tr) shape: (1095, 1)

Test Features (X_te) shape: (485, 545)
Test Target (Y_te) shape: (485, 1)


In [7]:
# Decoding location names and keywords
loc_names = [str(x[0]) if isinstance(x, np.ndarray) else str(x) for x in data['flu_locs'].ravel()]
keywords = [str(w[0]) if isinstance(w, np.ndarray) else str(w) for w in data['flu_keywords'].ravel()]

print(f"First 5 regions: {loc_names[:5]}")
print(f"Total regions: {len(loc_names)}, Total keywords: {len(keywords)}")

# Extract features and targets for region 0
X_region_0 = data['flu_X_tr'][0, 0]
Y_region_0 = data['flu_Y_tr'][0, 0]

# If X is sparse, convert to dense for inspection; 
# if dense, check shape
# They're all increcibly sparse b/c a tweet will only have a few of the many keywords
if hasattr(X_region_0, 'toarray'):
    X_region_0 = X_region_0.toarray()

print("Region 0 Features (X) shape:", X_region_0.shape)  # typically (T_weeks, 525)
print("Region 0 Target (Y) shape:", Y_region_0.shape)      # typically (T_weeks, 1)

First 5 regions: ['wyoming', 'colorado', 'nebraska', 'washington', 'rhode island']
Total regions: 48, Total keywords: 525
Region 0 Features (X) shape: (1095, 545)
Region 0 Target (Y) shape: (1095, 1)


# Algorithm 2: Logistic Regression


We can take the array of locations and apply logistic regression to train our model. As our data for each will be categorized as inactive (0) or active (1) for outbreaks, and our algorithm will learn what the probability of a location being active (1) is depending on the features.

In [8]:
#Using data from all locations for our model

num_locations = data['flu_X_tr'].shape[1]

def prep_all(mat):
    all_locs = []

    for i in range(num_locations):
        arr = mat[0, i]
        arr = arr.toarray() if hasattr(arr, 'toarray') else np.array(arr)

        arr = arr[:, :525]

        all_locs.append(arr)

    return np.vstack(all_locs)

def prep_all_targets(mat):
    all_targets = []

    for i in range(num_locations):
        arr = mat[0, i]
        arr = arr.toarray() if hasattr(arr, 'toarray') else np.array(arr)

        all_targets.append((arr.ravel() > 0).astype(int))

    return np.hstack(all_targets)

X_train_all = prep_all(data['flu_X_tr'])
y_train_all = prep_all_targets(data['flu_Y_tr'])

X_test_all = prep_all(data['flu_X_te'])
y_test_all = prep_all_targets(data['flu_Y_te'])

print("Training features:", X_train_all.shape)
print("Training targets:", y_train_all.shape)
print("Testing features:", X_test_all.shape)
print("Testing targets:", y_test_all.shape)

#How many of each target class do we have
print("\nTraining target counts: ", np.unique(y_train_all, return_counts=True))
print("Testing target counts: ", np.unique(y_test_all, return_counts=True))




Training features: (52560, 525)
Training targets: (52560,)
Testing features: (23280, 525)
Testing targets: (23280,)

Training target counts:  (array([0, 1]), array([49655,  2905]))
Testing target counts:  (array([0, 1]), array([20998,  2282]))


Our goal is to split our results into inactive or active. The dataset already has testing and training data, so we split the training data to give us validation data to work with as well.

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(X_train_all, y_train_all, test_size=0.25, random_state=42, stratify=y_train_all)

model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))

model.fit(X_train, y_train)

y_val_prob = model.predict_proba(X_val)[:, 1]
y_val_pred = model.predict(X_val)

#print accuracy
print("Validation Accuracy:", accuracy_score(y_val, y_val_pred))

#print classification report
print("Classification Report:\n", classification_report(y_val, y_val_pred, zero_division=0))

#print confusionn matrix to compare to other results
print("Confusion Matrix:\n", confusion_matrix(y_val, y_val_pred))

#print roc auc score
print("ROC AUC Score:", roc_auc_score(y_val, y_val_prob))


Validation Accuracy: 0.9472602739726027
Classification Report:
               precision    recall  f1-score   support

           0       0.95      0.99      0.97     12414
           1       0.57      0.19      0.29       726

    accuracy                           0.95     13140
   macro avg       0.76      0.59      0.63     13140
weighted avg       0.93      0.95      0.93     13140

Confusion Matrix:
 [[12306   108]
 [  585   141]]
ROC AUC Score: 0.7631157459741756


With the validation data, the model does well at identifying inactive cases but not for active cases, which is more important for the objective. The low ROC score indicates that the model does not distinguishes between outbreak and non-outbreak classes very well. With such little cases where the site is active, this makes it tricky to detect when an outbreak will occur. However, it is important to remember our goal is to predict outbreaks.

In [11]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score

y_test_prob = model.predict_proba(X_test_all)[:, 1]

y_pred = model.predict(X_test_all)

#accuracy score
accuracy = accuracy_score(y_test_all, y_pred)

#classification report
print("Classification Report:\n", classification_report(y_test_all, y_pred, zero_division=0))

#confusion matrix
print("Confusion Matrix:\n", confusion_matrix(y_test_all, y_pred))

#roc score 
roc_auc = roc_auc_score(y_test_all, y_test_prob)
print("ROC AUC Score:", roc_auc)




Classification Report:
               precision    recall  f1-score   support

           0       0.91      0.99      0.95     20998
           1       0.59      0.11      0.18      2282

    accuracy                           0.91     23280
   macro avg       0.75      0.55      0.57     23280
weighted avg       0.88      0.91      0.87     23280

Confusion Matrix:
 [[20826   172]
 [ 2038   244]]
ROC AUC Score: 0.7492859488558612
